# Personal AI Entertainment Recommender — MovieLens 600
**Sidy Badiane | Group 7 | MiniLM + ALS**

Use the small GroupLens education/development archive supplied with this project. Its actual size is 610 users, 100,836 ratings and 9,742 catalog titles. We sample **exactly 600 users**, seed 42, preserving original IDs. No synthetic viewers are created.

**Know what the data means:** these are real star ratings, not playback histories. The catalog has titles/genres and some user tags, but no plots, verified streaming platforms, or curated TV-show collection. MiniLM uses those available fields. Optional description and TV-title CSVs are supported; nothing is invented. “Because You Watched” is an item-similarity panel, not a claim that a MovieLens user watched a title at the rating time.

**Panels:** three-favorite onboarding; MiniLM semantic search; Because You Watched (80% MiniLM, 20% genres); Top Picks (explicit ALS + MiniLM + genres); Most Popular (historical recent raters + smoothed ratings, optionally observed views and release demand).

**Kaggle:** attach `ml-latest-small.zip` or its extracted four CSV files, enable Internet for initial installation/model download, and Run All. CPU is sufficient. ALS uses local NumPy/SciPy least-squares updates; Java and Spark are not required. The first run can take several minutes. This notebook is self-contained; uploading the separate Python file is not required.

In [1]:
import os, sys, subprocess, importlib.util, shutil
os.environ.setdefault('OMP_NUM_THREADS','2')
os.environ.setdefault('OPENBLAS_NUM_THREADS','2')
dependencies = {'sentence_transformers':'sentence-transformers',
                'ipywidgets':'ipywidgets'}
for module, package in dependencies.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
from IPython.display import display


## Engine implementation
This cell defines the complete recommender. Subsequent cells load data, train, evaluate and show the interactive controls.

In [2]:
"""MovieLens MiniLM + CPU explicit ALS educational recommender (600 sampled users)."""
from pathlib import Path
import hashlib
import json
import os
import zipfile
import numpy as np
import pandas as pd

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'


def load_movielens(source, users=600, seed=42):
    source = Path(source)
    names = ['movies', 'ratings', 'tags', 'links']
    if source.is_file():
        with zipfile.ZipFile(source) as archive:
            frames = {}
            for name in names:
                matches = [p for p in archive.namelist() if p.endswith('/'+name+'.csv') or p == name+'.csv']
                if len(matches) != 1:
                    raise ValueError(f'Expected exactly one {name}.csv in ZIP.')
                with archive.open(matches[0]) as stream:
                    frames[name] = pd.read_csv(stream)
    else:
        frames = {name: pd.read_csv(source/(name+'.csv')) for name in names}
    ids = np.sort(frames['ratings'].userId.unique())
    if len(ids) < users:
        raise ValueError(f'Need {users} users; only {len(ids)} found.')
    selected = np.sort(np.random.default_rng(seed).choice(ids, users, replace=False))
    for name in ['ratings', 'tags']:
        frames[name] = frames[name][frames[name].userId.isin(selected)].copy()
        frames[name]['at'] = pd.to_datetime(frames[name].timestamp, unit='s', utc=True)
    ratings = frames['ratings']
    if ratings[['userId','movieId','rating','timestamp']].isna().any().any():
        raise ValueError('Missing rating fields.')
    if ratings.duplicated(['userId','movieId']).any() or not ratings.rating.between(.5,5).all():
        raise ValueError('Expected unique user/movie pairs and ratings in 0.5–5.')
    if not set(ratings.movieId) <= set(frames['movies'].movieId):
        raise ValueError('Unknown movie IDs in ratings.')
    frames['selected_users'] = pd.DataFrame({'userId':selected})
    return frames


def normalize_score(values):
    values = np.asarray(values, dtype=float)
    lo, hi = values.min(), values.max()
    return (values-lo)/(hi-lo) if hi > lo else np.zeros_like(values)


class EntertainmentRecommender:
    def __init__(self, data, output='outputs', descriptions=None, extra_media=None,
                 watch_events=None, releases=None):
        self.out = Path(output)
        self.out.mkdir(parents=True, exist_ok=True)
        self.data = data
        self.ratings = data['ratings'].copy()
        self.catalog = data['movies'].copy()
        self.catalog['media_type'] = 'Movie'
        self.catalog['description'] = ''
        if descriptions is not None:
            d = pd.read_csv(descriptions)
            if not {'movieId','description'} <= set(d) or d.movieId.duplicated().any():
                raise ValueError('descriptions.csv needs unique movieId and description columns.')
            if not set(d.movieId) <= set(self.catalog.movieId):
                raise ValueError('Description IDs must match MovieLens IDs.')
            self.catalog['description'] = self.catalog.movieId.map(d.set_index('movieId').description).fillna('')
        if extra_media is not None:
            extra = pd.read_csv(extra_media)
            required = {'movieId','title','genres','description','media_type'}
            if not required <= set(extra) or extra[list(required)].isna().any().any():
                raise ValueError('extra_media.csv needs movieId,title,genres,description,media_type.')
            if not extra.media_type.isin(['Movie','TV Show']).all():
                raise ValueError('media_type must be Movie or TV Show.')
            if not (extra.movieId < 0).all() or extra.movieId.duplicated().any():
                raise ValueError('Extra titles need unique negative integer movieId values.')
            if not np.equal(extra.movieId, extra.movieId.astype(int)).all():
                raise ValueError('movieId must be integer.')
            self.catalog = pd.concat([self.catalog,extra[list(required)]],ignore_index=True)
        self.catalog = self.catalog.sort_values('movieId').reset_index(drop=True)
        if self.catalog.movieId.duplicated().any():
            raise ValueError('Duplicate catalog movieId.')
        self.ids = self.catalog.movieId.to_numpy(dtype=int)
        self.index = {int(mid):i for i,mid in enumerate(self.ids)}
        self.genre_sets = [set(str(g).split('|'))-{'(no genres listed)',''} for g in self.catalog.genres]
        from sklearn.preprocessing import MultiLabelBinarizer
        self.G = MultiLabelBinarizer().fit_transform(self.genre_sets).astype(np.float32)
        self.genre_counts = self.G.sum(axis=1)
        self.profile_cache = {}
        self.watch = None
        self.releases = None
        if watch_events is not None:
            w = pd.read_csv(watch_events)
            required = {'eventId','userId','movieId','unitId','watchedAt','completion'}
            if not required <= set(w) or w[list(required)].isna().any().any():
                raise ValueError('watch_events.csv has missing fields; see template.')
            if w.eventId.duplicated().any() or not w.completion.between(0,1).all():
                raise ValueError('Watch event IDs must be unique; completion must be 0–1.')
            if not set(w.movieId) <= set(self.ids):
                raise ValueError('Watch events contain unknown media IDs.')
            w['at'] = pd.to_datetime(w.watchedAt,utc=True,errors='raise')
            w['unitId'] = w.unitId.astype(str)
            self.watch = w
        if releases is not None:
            r = pd.read_csv(releases)
            if not {'movieId','unitId','releasedAt'} <= set(r):
                raise ValueError('releases.csv needs movieId,unitId,releasedAt.')
            if r[['movieId','unitId','releasedAt']].isna().any().any() or r.duplicated(['movieId','unitId']).any():
                raise ValueError('Release keys must be unique and complete.')
            if not set(r.movieId) <= set(self.ids):
                raise ValueError('Unknown release movieId.')
            r['released'] = pd.to_datetime(r.releasedAt,utc=True,errors='raise')
            r['unitId'] = r.unitId.astype(str)
            self.releases = r
        self.encoder = None
        self.U, self.V = {}, {}
        self.asof = self.ratings['at'].max()
        self.catalog['platform'] = 'Not provided by MovieLens'

    def encode_catalog(self, asof):
        from sentence_transformers import SentenceTransformer
        if self.encoder is None:
            self.encoder = SentenceTransformer(MODEL_NAME)
        # Only tags available before the snapshot are used during evaluation.
        tags = self.data['tags']
        tags = tags[tags['at'] <= asof].copy()
        tags['tag'] = tags.tag.fillna('').astype(str).str.strip()
        tags = tags.groupby('movieId').tag.apply(lambda s:'; '.join(sorted(set(s))[:12]))
        self.catalog['tags'] = self.catalog.movieId.map(tags).fillna('')
        text = ('Title: '+self.catalog.title.fillna('')+'. Genres: '+
                self.catalog.genres.fillna('').str.replace('|',', ',regex=False)+
                '. Description: '+self.catalog.description.fillna('')+
                '. Tags: '+self.catalog.tags)
        key = hashlib.sha256((MODEL_NAME+'\n'+'\n'.join(text)).encode()).hexdigest()[:20]
        cache = self.out/f'embeddings_{key}.npy'
        if cache.exists():
            self.E = np.load(cache)
        else:
            self.E = self.encoder.encode(text.tolist(),batch_size=64,normalize_embeddings=True,
                                         show_progress_bar=True,convert_to_numpy=True)
            np.save(cache,self.E)
        if self.E.shape != (len(self.catalog),384) or not np.isfinite(self.E).all():
            raise ValueError('Invalid MiniLM embedding cache.')
        self.embedding_snapshot = asof
        self.profile_cache = {}

    def fit(self, history, asof, rank=24, reg=.1):
        from scipy.linalg import solve
        self.history = history[history['at'] <= asof].copy()
        self.asof = asof
        self.profile_cache = {}
        if self.history.empty or rank < 1 or reg <= 0:
            raise ValueError('ALS requires history, positive rank and regularization.')
        # Observed-star least squares with training-only baseline biases.
        # Missing user/movie pairs are not zeros and never enter this loss.
        h=self.history.copy()
        self.global_mean=float(h.rating.mean())
        residual=h.rating-self.global_mean
        ug=residual.groupby(h.userId).agg(['sum','count'])
        self.user_bias=(ug['sum']/(ug['count']+10)).to_dict()
        residual=residual-h.userId.map(self.user_bias)
        ig=residual.groupby(h.movieId).agg(['sum','count'])
        self.item_bias={self.index[int(mid)]:float(row['sum']/(row['count']+10))
                        for mid,row in ig.iterrows()}
        h['item']=h.movieId.map(self.index)
        h['target']=residual-h['item'].map(self.item_bias)
        uid=sorted(h.userId.unique()); iid=sorted(h['item'].unique())
        umap={u:j for j,u in enumerate(uid)}; imap={i:j for j,i in enumerate(iid)}
        h['u']=h.userId.map(umap); h['i']=h['item'].map(imap)
        by_user=[(int(u),g.i.to_numpy(),g.target.to_numpy()) for u,g in h.groupby('u')]
        by_item=[(int(i),g.u.to_numpy(),g.target.to_numpy()) for i,g in h.groupby('i')]
        rng=np.random.default_rng(42)
        P=rng.normal(0,.1,(len(uid),int(rank)))
        Q=rng.normal(0,.1,(len(iid),int(rank)))
        eye=np.eye(int(rank)); losses=[]
        for iteration in range(10):
            for u,items,target in by_user:
                A=Q[items]
                P[u]=solve(A.T@A+float(reg)*len(items)*eye,A.T@target,assume_a='pos',check_finite=False)
            for i,people,target in by_item:
                A=P[people]
                Q[i]=solve(A.T@A+float(reg)*len(people)*eye,A.T@target,assume_a='pos',check_finite=False)
            prediction=np.sum(P[h.u.to_numpy()]*Q[h.i.to_numpy()],axis=1)
            objective=float(np.sum((h.target.to_numpy()-prediction)**2))
            objective+=float(reg)*sum(len(items)*float(P[u]@P[u]) for u,items,_ in by_user)
            objective+=float(reg)*sum(len(people)*float(Q[i]@Q[i]) for i,people,_ in by_item)
            losses.append(objective)
        if not np.isfinite(P).all() or not np.isfinite(Q).all():
            raise ValueError('ALS produced non-finite factors.')
        self.als_training_objective=losses
        self.U={int(u):P[j] for j,u in enumerate(uid)}
        self.V={int(i):Q[j] for j,i in enumerate(iid)}
        self.factor_items = np.array(sorted(self.V),dtype=int)
        self.factor_matrix = np.vstack([self.V[i] for i in self.factor_items])
        self.seen = self.history.groupby('userId').movieId.apply(set).to_dict()
        self.groups = {int(u):g for u,g in self.history.groupby('userId')}
        self.pop_table = self.popularity(self.history,asof)
        self.pop = self.pop_table.score.to_numpy()
        return self

    def popularity(self, history, asof, use_watches=False, window=90):
        h = history[history['at'] <= asof]
        means = h.groupby('movieId').rating.agg(['mean','count'])
        global_mean = float(h.rating.mean()) if len(h) else 3.0
        count = self.catalog.movieId.map(means['count']).fillna(0).to_numpy()
        avg = self.catalog.movieId.map(means['mean']).fillna(global_mean).to_numpy()
        quality = (count*avg+20*global_mean)/(count+20)/5
        # Rating timestamps measure rating activity, never watch/release time.
        base = h
        label = 'Historical unique raters (not viewers)'
        if use_watches and self.watch is not None:
            base = self.watch[(self.watch['at'] <= asof)&(self.watch.completion >= .2)]
            label = 'Observed unique viewers'
        recent = base[(base['at'] <= asof)&(base['at'] >= asof-pd.Timedelta(days=window))]
        counts = recent.groupby('movieId').userId.nunique()
        recency = self.catalog.movieId.map(counts).fillna(0).to_numpy(dtype=float)
        keen = np.zeros(len(self.catalog))
        keen_enabled = use_watches and self.watch is not None and self.releases is not None
        if keen_enabled:
            known = self.releases[self.releases.released <= asof]
            joined = base.merge(known,on=['movieId','unitId'],how='inner',validate='many_to_one')
            lag = (joined['at']-joined.released).dt.total_seconds()/3600
            early = joined[(lag >= 0)&(lag <= 48)].copy()
            early['weight'] = np.power(.5,(asof-early.released).dt.total_seconds()/86400/30)
            # One viewer counts at most once per show, even across episodes/replays.
            demand = early.groupby(['movieId','userId']).weight.max().groupby('movieId').sum()
            keen = self.catalog.movieId.map(demand).fillna(0).to_numpy()
        recency_scaled = normalize_score(np.log1p(recency))
        score = .7*recency_scaled+.3*quality
        if keen_enabled:
            score = .55*recency_scaled+.25*quality+.2*normalize_score(np.log1p(keen))
        out = self.catalog[['movieId','title']].copy()
        out['score'] = score
        out['recent_unique_users'] = recency.astype(int)
        out['rating_count'] = count.astype(int)
        out['bayesian_rating'] = quality*5
        out['keen_score'] = keen if keen_enabled else np.nan
        out['signal'] = label
        out['as_of'] = str(asof)
        return out

    def _genre_scores(self, seeds):
        if not seeds:
            return np.zeros(len(self.catalog))
        overlap = self.G @ self.G[seeds].T
        union = self.genre_counts[:,None]+self.genre_counts[seeds][None,:]-overlap
        return np.divide(overlap,union,out=np.zeros_like(overlap),where=union>0).mean(axis=1)

    def _content_profile(self,user_id=None,favorites=()):
        key = (user_id,tuple(favorites))
        if key in self.profile_cache: return self.profile_cache[key]
        seeds = [self.index[int(mid)] for mid in favorites]
        weights = [1.0]*len(seeds)
        h = self.groups.get(user_id,pd.DataFrame())
        if len(h):
            liked = h[h.rating >= 4].sort_values('at').tail(50)
            seeds += [self.index[int(mid)] for mid in liked.movieId]
            weights += (liked.rating-3).tolist()
        if self.watch is not None and user_id is not None:
            watched = self.watch[(self.watch.userId == user_id)&(self.watch['at'] <= self.asof)&
                                 (self.watch.completion >= .8)].drop_duplicates('movieId').tail(50)
            seeds += [self.index[int(mid)] for mid in watched.movieId]
            weights += watched.completion.tolist()
        if not seeds:
            return np.zeros(len(self.catalog)),False
        profile = np.average(self.E[seeds],axis=0,weights=weights)
        profile /= max(np.linalg.norm(profile),1e-8)
        semantic = np.clip(self.E@profile,0,1)
        result = (.85*semantic+.15*self._genre_scores(seeds),True)
        self.profile_cache[key] = result
        return result

    def _scores(self,user_id=None,favorites=(),method='hybrid'):
        favorites = list(favorites)
        if not set(favorites) <= set(self.index):
            raise ValueError('Unknown favorite movieId.')
        if method == 'popular': return self.pop.copy(),'Popularity baseline'
        content,has_content = self._content_profile(user_id,favorites)
        if method == 'content': return (content if has_content else self.pop.copy()),'MiniLM + genre profile'
        if user_id not in self.U:
            return (content if has_content else self.pop.copy()),'Favorite-title/content fallback' if has_content else 'Popularity fallback'
        raw = (self.global_mean+self.user_bias.get(user_id,0)+
               np.array([self.item_bias.get(i,0) for i in self.factor_items])+
               self.factor_matrix@self.U[user_id])
        # Explicit ALS predictions are mapped to the rating scale; not probabilities.
        als = np.clip(raw,.5,5)/5
        scores = content.copy() if has_content else self.pop.copy()
        if method == 'als':
            scores[self.factor_items] = als
            return scores,'ALS; content/popularity for untrained titles'
        scores[self.factor_items] = (.7*als+.25*content[self.factor_items]+.05*self.pop[self.factor_items]
                                    if has_content else .95*als+.05*self.pop[self.factor_items])
        return scores,'ALS + MiniLM + genres + popularity'

    def _rank(self,scores,n=10,exclude=(),diversity=.08,media_type=None):
        if not isinstance(n,int) or n < 1: raise ValueError('n must be a positive integer.')
        allowed = np.ones(len(self.catalog),dtype=bool)
        for mid in exclude:
            if mid in self.index: allowed[self.index[mid]]=False
        if media_type is not None:
            if media_type not in ['Movie','TV Show']: raise ValueError('Unknown media_type.')
            allowed &= self.catalog.media_type.eq(media_type).to_numpy()
        pool = [int(i) for i in np.argsort(-scores,kind='stable') if allowed[i]][:max(150,n*10)]
        selected = []
        while pool and len(selected) < n:
            if not selected or diversity == 0:
                pick=pool[0]
            else:
                similar=(self.E[pool]@self.E[selected].T).max(axis=1)
                pick=pool[int(np.argmax((1-diversity)*scores[pool]-diversity*similar))]
            selected.append(pick); pool.remove(pick)
        return selected

    def _table(self,indices,scores,reason):
        out=self.catalog.iloc[indices][['movieId','title','media_type','genres','description','tags','platform']].copy()
        out['score']=np.round(scores[indices],4)
        out['reason']=reason
        out['text_source']=np.where(out.description.str.len()>0,'Provided description + metadata','Title + genres + tags; no plot supplied')
        return out.reset_index(drop=True)

    def top_picks(self,user_id=None,favorites=(),n=10,media_type=None):
        favorites=list(favorites)
        scores,reason=self._scores(user_id,favorites)
        excluded=self.seen.get(user_id,set()) | set(favorites)
        if self.watch is not None and user_id is not None:
            excluded |= set(self.watch.loc[(self.watch.userId==user_id)&(self.watch['at']<=self.asof),'movieId'])
        indices=self._rank(scores,n,excluded,media_type=media_type)
        return self._table(indices,scores,reason)

    def from_three_favorites(self,movie_ids,n=10):
        if len(movie_ids)!=3 or len(set(movie_ids))!=3:
            raise ValueError('Choose exactly three different movie IDs.')
        return self.top_picks(favorites=movie_ids,n=n)

    def because_you_watched(self,movie_id,user_id=None,n=10,media_type=None):
        if movie_id not in self.index: raise ValueError('Unknown movie ID.')
        seed=self.index[movie_id]
        semantic=np.clip(self.E@self.E[seed],0,1)
        genre=self._genre_scores([seed])
        scores=.8*semantic+.2*genre
        excluded=self.seen.get(user_id,set())|{movie_id}
        if self.watch is not None and user_id is not None:
            excluded |= set(self.watch.loc[(self.watch.userId==user_id)&(self.watch['at']<=self.asof),'movieId'])
        indices=self._rank(scores,n,excluded,media_type=media_type)
        out=self._table(indices,scores,'MiniLM text similarity + genre overlap')
        out['shared_genres']=[', '.join(sorted(self.genre_sets[seed]&self.genre_sets[i])) for i in indices]
        return out

    def search(self,request,n=10,media_type=None):
        if not request.strip(): raise ValueError('Enter a search request.')
        q=self.encoder.encode(request,normalize_embeddings=True)
        scores=np.clip(self.E@q,0,1)
        return self._table(self._rank(scores,n,diversity=0,media_type=media_type),scores,'MiniLM semantic search')

    def find_titles(self,text,n=20):
        return self.catalog.loc[self.catalog.title.str.contains(text,case=False,regex=False),['movieId','title','media_type','genres']].head(n)

    def record_watch(self,user_id,movie_id,completion,unit_id='movie',watched_at=None):
        """Store user-reported viewing. Not automatically observed streaming telemetry."""
        import uuid
        if movie_id not in self.index or not np.isfinite(completion) or not 0 <= completion <= 1:
            raise ValueError('Use a valid movieId and completion between 0 and 1.')
        at = pd.Timestamp.now(tz='UTC') if watched_at is None else pd.to_datetime(watched_at,utc=True)
        if pd.isna(at) or at > pd.Timestamp.now(tz='UTC'):
            raise ValueError('Viewing time must be valid and cannot be in the future.')
        row = pd.DataFrame([{'eventId':str(uuid.uuid4()),'userId':int(user_id),
                             'movieId':int(movie_id),'unitId':str(unit_id),'watchedAt':at.isoformat(),
                             'completion':float(completion),'at':at}])
        self.watch = row if self.watch is None else pd.concat([self.watch,row],ignore_index=True)
        self.asof = max(self.asof,at)
        self.profile_cache = {}
        self.watch.drop(columns='at').to_csv(self.out/'collected_watch_events.csv',index=False)
        return 'Saved user-reported viewing; content profile updated. ALS requires a later batch update.'

    def most_popular(self,n=10):
        asof=self.watch['at'].max() if self.watch is not None and len(self.watch) else self.asof
        out=self.popularity(self.history,asof,use_watches=self.watch is not None)
        eligible=out.recent_unique_users.gt(0) | out.rating_count.gt(0)
        return out[eligible].sort_values(['score','rating_count'],ascending=False).head(n).reset_index(drop=True)

    def evaluate(self,future,k=10):
        relevant=future[future.rating>=4].groupby('userId').movieId.apply(set)
        rows=[]
        for method in ['popular','content','als','hybrid']:
            recall=[]; ndcg=[]; chosen=set(); cold=0; targets=0
            for user,truth in relevant.items():
                truth=truth-self.seen.get(user,set())
                if not truth: continue
                scores,_=self._scores(int(user),method=method)
                ids=self._rank(scores,k,self.seen.get(user,set()),diversity=.08 if method=='hybrid' else 0)
                mids=[int(self.ids[i]) for i in ids]; chosen.update(mids)
                hits=[int(mid in truth) for mid in mids]
                recall.append(sum(hits)/len(truth))
                dcg=sum(hit/np.log2(j+2) for j,hit in enumerate(hits))
                ideal=sum(1/np.log2(j+2) for j in range(min(k,len(truth))))
                ndcg.append(dcg/ideal)
                cold+=sum(user not in self.U or self.index[mid] not in self.V for mid in truth)
                targets+=len(truth)
            rows.append({'model':method,'recall_at_10':float(np.mean(recall)) if recall else 0,
                         'ndcg_at_10':float(np.mean(ndcg)) if ndcg else 0,
                         'evaluated_users':len(recall),'catalog_coverage':len(chosen)/len(self.catalog),
                         'cold_target_fraction':cold/max(targets,1)})
        return pd.DataFrame(rows)

    def rating_rmse(self,future):
        actual=[]; predicted=[]
        for r in future.itertuples():
            i=self.index[int(r.movieId)]
            if r.userId in self.U and i in self.V:
                actual.append(r.rating)
                predicted.append(float(self.global_mean+self.user_bias.get(r.userId,0)+
                                       self.item_bias.get(i,0)+self.U[r.userId]@self.V[i]))
        return {'rmse':float(np.sqrt(np.mean((np.array(actual)-np.array(predicted))**2))) if actual else None,
                'prediction_coverage':len(actual)/max(1,len(future))}

    def train_evaluate(self):
        ordered=self.ratings.sort_values(['at','userId','movieId'])
        t1=ordered.iloc[int(len(ordered)*.7)]['at']; t2=ordered.iloc[int(len(ordered)*.85)]['at']
        train=ordered[ordered['at']<=t1]; valid=ordered[(ordered['at']>t1)&(ordered['at']<=t2)]
        test=ordered[ordered['at']>t2]
        self.encode_catalog(t1)
        trials=[]
        for rank,reg in [(16,.1),(32,.1),(32,.2)]:
            print(f'Training validation model rank={rank}, regularization={reg}',flush=True)
            self.fit(train,t1,rank,reg)
            quality=self.evaluate(valid)
            value=quality.loc[quality.model=='hybrid','ndcg_at_10'].iloc[0]
            trials.append({'rank':rank,'regularization':reg,'validation_hybrid_ndcg_at_10':value,**self.rating_rmse(valid)})
        self.tuning=pd.DataFrame(trials).sort_values('validation_hybrid_ndcg_at_10',ascending=False,kind='stable')
        best=self.tuning.iloc[0]
        self.best={'rank':int(best['rank']),'reg':float(best['regularization'])}
        self.encode_catalog(t2)
        self.fit(ordered[ordered['at']<=t2],t2,**self.best)
        self.test_metrics=self.evaluate(test)
        self.test_rmse=self.rating_rmse(test)
        self.split_info={'train_cutoff':str(t1),'validation_cutoff':str(t2),
                         'train_rows':len(train),'validation_rows':len(valid),'test_rows':len(test),
                         'test_rating_metrics':self.test_rmse,
                         'evaluation_note':'Retrospective fixed catalog; tags and ratings obey global time cutoffs. Rating dates are not watch dates. Static metadata availability is not time reconstructed.'}
        self.tuning.to_csv(self.out/'validation_results.csv',index=False)
        self.test_metrics.to_csv(self.out/'test_metrics.csv',index=False)
        (self.out/'evaluation_details.json').write_text(json.dumps(self.split_info,indent=2))
        # Refit for demonstration only after the held-out metrics have been recorded.
        self.encode_catalog(ordered['at'].max())
        self.fit(ordered,ordered['at'].max(),**self.best)
        return self.test_metrics

    def export(self,user_id=1):
        import sqlite3
        self.data['selected_users'].to_csv(self.out/'selected_600_users.csv',index=False)
        panels={'top_picks':self.top_picks(user_id),
                'three_favorites':self.from_three_favorites([1,260,2571]),
                'because_you_watched':self.because_you_watched(1),
                'most_popular':self.most_popular()}
        for name,table in panels.items(): table.to_csv(self.out/(name+'.csv'),index=False)
        with sqlite3.connect(self.out/'entertainment_recommender.sqlite') as db:
            self.catalog.to_sql('catalog',db,if_exists='replace',index=False)
            self.ratings.to_sql('ratings',db,if_exists='replace',index=False)
            self.data['selected_users'].to_sql('selected_users',db,if_exists='replace',index=False)
            for name,table in panels.items(): table.to_sql(name,db,if_exists='replace',index=False)
            self.test_metrics.to_sql('test_metrics',db,if_exists='replace',index=False)
        np.save(self.out/'catalog_embeddings.npy',self.E)
        self.catalog[['movieId','title']].to_csv(self.out/'embedding_id_mapping.csv',index=False)
        np.savez_compressed(self.out/'als_factors.npz',user_ids=np.array(sorted(self.U)),
                            user_factors=np.vstack([self.U[u] for u in sorted(self.U)]),
                            item_indices=self.factor_items,item_factors=self.factor_matrix,
                            global_mean=self.global_mean,
                            user_bias=np.array([self.user_bias[u] for u in sorted(self.U)]),
                            item_bias=np.array([self.item_bias[i] for i in self.factor_items]))
        return panels


## Locate the dataset and optional metadata
Optional inputs (leave `None` for the MovieLens-only run):
- descriptions.csv: `movieId,description` using original MovieLens IDs.
- extra_media.csv: `movieId,title,genres,description,media_type` using unique negative IDs; genres separated with `|`; media_type `TV Show` or `Movie`. These items use content fallback until interactions are collected.
- watch_events.csv: `eventId,userId,movieId,unitId,watchedAt,completion` (UTC; completion 0–1).
- releases.csv: `movieId,unitId,releasedAt` (UTC; one row per movie/episode release).

Do not substitute a rating timestamp or the year in a title for a watch/release timestamp. No third-party API key is required for the base project.

In [3]:
def discover_data():
    roots = [Path('/kaggle/input'), Path.cwd()]
    for root in roots:
        if not root.exists(): continue
        for folder in root.rglob('ratings.csv'):
            if all((folder.parent/(name+'.csv')).exists() for name in ['movies','ratings','tags','links']):
                return folder.parent
        hits = list(root.rglob('ml-latest-small.zip'))
        if hits: return hits[0]
    raise FileNotFoundError('Attach ml-latest-small.zip or the extracted MovieLens CSV folder.')

SOURCE = discover_data()
OUTPUT = Path('/kaggle/working/movielens_outputs') if Path('/kaggle').exists() else Path('outputs')
DESCRIPTIONS = None
EXTRA_MEDIA = None
WATCH_EVENTS = None
RELEASES = None
data = load_movielens(SOURCE, users=600, seed=42)
print('Source:',SOURCE)
print('Sampled users:',data['ratings'].userId.nunique())
print('Real ratings:',len(data['ratings']))
print('Catalog titles:',len(data['movies']))
print('Tags from sampled users:',len(data['tags']))
engine = EntertainmentRecommender(data, OUTPUT, descriptions=DESCRIPTIONS,
                                  extra_media=EXTRA_MEDIA,watch_events=WATCH_EVENTS,releases=RELEASES)
display(data['movies'].head())


Source: /kaggle/input/datasets/sidybadiane/ml-latest-small/ml-latest-small
Sampled users: 600
Real ratings: 99609
Catalog titles: 9742
Tags from sampled users: 3683


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
cutoff = data["ratings"]["at"].max()
engine.encode_catalog(cutoff)
engine.fit(data["ratings"], cutoff, rank=16, reg=0.1)

display(engine.because_you_watched(1, n=5))
display(engine.top_picks(user_id=1, n=5))
display(engine.most_popular(n=5))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/153 [00:00<?, ?it/s]

,movieId,title,media_type,genres,description,tags,platform,score,reason,text_source,shared_genres
0,3114,Toy Story 2 (1999),Movie,Adventure|Animation|Children|Comedy|Fantasy,,Disney; Pixar; Tom Hanks; animation; funny; or...,Not provided by MovieLens,0.8186,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy, Fantasy"
1,78499,Toy Story 3 (2010),Movie,Adventure|Animation|Children|Comedy|Fantasy|IMAX,,,Not provided by MovieLens,0.7582,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy, Fantasy"
2,2355,"Bug's Life, A (1998)",Movie,Adventure|Animation|Children|Comedy,,Pixar,Not provided by MovieLens,0.7110,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy"
3,239,"Goofy Movie, A (1995)",Movie,Animation|Children|Comedy|Romance,,,Not provided by MovieLens,0.6771,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Animation, Children, Comedy"
4,145935,"Peanuts Movie, The (2015)",Movie,Adventure|Animation|Children|Comedy,,,Not provided by MovieLens,0.6756,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy"


,movieId,title,media_type,genres,description,tags,platform,score,reason,text_source
0,58559,"Dark Knight, The (2008)",Movie,Action|Crime|Drama|IMAX,,dark; gritty; psychology; superhero,Not provided by MovieLens,0.8970,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
1,858,"Godfather, The (1972)",Movie,Crime|Drama,,Mafia,Not provided by MovieLens,0.8934,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
2,318,"Shawshank Redemption, The (1994)",Movie,Crime|Drama,,Morgan Freeman; Stephen King; prison; wrongful...,Not provided by MovieLens,0.8751,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
3,177593,"Three Billboards Outside Ebbing, Missouri (2017)",Movie,Crime|Drama,,,Not provided by MovieLens,0.8702,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
4,56782,There Will Be Blood (2007),Movie,Drama|Western,,atmospheric; cerebral; character study; gritty...,Not provided by MovieLens,0.8783,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied


,movieId,title,score,recent_unique_users,rating_count,bayesian_rating,keen_score,signal,as_of
0,79132,Inception (2010),0.939226,10,140,3.987107,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
1,2571,"Matrix, The (1999)",0.921205,9,272,4.150470,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
2,187593,Deadpool 2 (2018),0.918320,10,12,3.638661,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
3,7153,"Lord of the Rings: The Return of the King, The...",0.915722,9,182,4.059095,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
4,4993,"Lord of the Rings: The Fellowship of the Ring,...",0.884041,8,196,4.043691,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00


## Train and measure quality
Global chronological 70% / 15% / 15% split (ties remain together). Rank and regularization are selected using validation hybrid NDCG@10. Test compares popularity, MiniLM content, ALS and the final diversity-reranked hybrid. Targets are ratings >=4. Recommendations exclude all titles the user already rated, including low-rated titles. Cold-start users/items use fallbacks and remain in ranking evaluation. RMSE is reported separately with known-factor coverage.

Tags are rebuilt at each time cutoff. The catalog is a fixed retrospective candidate set; metadata availability and movie release availability are NOT historically reconstructed. Hence these are classroom offline metrics, not a simulated historical launch or online success estimate. Favorite onboarding has no held-out metric because real onboarding choices are not in MovieLens. No claim of improvement over VRV is made.

In [5]:
metrics = engine.train_evaluate()
display(engine.tuning)
display(metrics)
print('Test rating RMSE and coverage:',engine.test_rmse)
print('Split details:',engine.split_info)


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

Training validation model rank=16, regularization=0.1
Training validation model rank=32, regularization=0.1
Training validation model rank=32, regularization=0.2


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

,rank,regularization,validation_hybrid_ndcg_at_10,rmse,prediction_coverage
2,32,0.2,0.096126,0.804853,0.071615
1,32,0.1,0.090426,0.817539,0.071615
0,16,0.1,0.089325,0.836122,0.071615


,model,recall_at_10,ndcg_at_10,evaluated_users,catalog_coverage,cold_target_fraction
0,popular,0.043559,0.208504,96,0.005132,0.838648
1,content,0.029597,0.170125,96,0.014165,0.838648
2,als,0.037303,0.188475,96,0.011702,0.838648
3,hybrid,0.046137,0.202261,96,0.013858,0.838648


Test rating RMSE and coverage: {'rmse': 0.8422596064758171, 'prediction_coverage': 0.13245432032661802}
Split details: {'train_cutoff': '2013-08-21 07:01:33+00:00', 'validation_cutoff': '2016-11-19 08:37:07+00:00', 'train_rows': 69727, 'validation_rows': 14941, 'test_rows': 14941, 'test_rating_metrics': {'rmse': 0.8422596064758171, 'prediction_coverage': 0.13245432032661802}, 'evaluation_note': 'Retrospective fixed catalog; tags and ratings obey global time cutoffs. Rating dates are not watch dates. Static metadata availability is not time reconstructed.'}


## Select three favorites
Use movie IDs to avoid confusing remakes with the same title. Example choices: Toy Story (1), Star Wars (260), and The Matrix (2571). The three selected favorites are excluded from the output. This personalizes immediately without using a fabricated viewing history.

In [6]:
FAVORITES = [1,260,2571]
display(engine.from_three_favorites(FAVORITES,n=10))
# Find IDs for different favorites:
display(engine.find_titles('Matrix'))


,movieId,title,media_type,genres,description,tags,platform,score,reason,text_source
0,2628,Star Wars: Episode I - The Phantom Menace (1999),Movie,Action|Adventure|Sci-Fi,,prequel; the Force,Not provided by MovieLens,0.7676,Favorite-title/content fallback,Title + genres + tags; no plot supplied
1,179819,Star Wars: The Last Jedi (2017),Movie,Action|Adventure|Fantasy|Sci-Fi,,,Not provided by MovieLens,0.7238,Favorite-title/content fallback,Title + genres + tags; no plot supplied
2,122886,Star Wars: Episode VII - The Force Awakens (2015),Movie,Action|Adventure|Fantasy|Sci-Fi|IMAX,,,Not provided by MovieLens,0.7137,Favorite-title/content fallback,Title + genres + tags; no plot supplied
3,122918,Guardians of the Galaxy 2 (2017),Movie,Action|Adventure|Sci-Fi,,fun,Not provided by MovieLens,0.7000,Favorite-title/content fallback,Title + genres + tags; no plot supplied
4,61160,Star Wars: The Clone Wars (2008),Movie,Action|Adventure|Animation|Sci-Fi,,,Not provided by MovieLens,0.7063,Favorite-title/content fallback,Title + genres + tags; no plot supplied
5,120799,Terminator Genisys (2015),Movie,Action|Adventure|Sci-Fi|Thriller,,,Not provided by MovieLens,0.6844,Favorite-title/content fallback,Title + genres + tags; no plot supplied
6,141544,Turbo Kid (2015),Movie,Action|Adventure|Sci-Fi,,,Not provided by MovieLens,0.6799,Favorite-title/content fallback,Title + genres + tags; no plot supplied
7,102880,After Earth (2013),Movie,Action|Adventure|Sci-Fi|IMAX,,,Not provided by MovieLens,0.6860,Favorite-title/content fallback,Title + genres + tags; no plot supplied
8,75416,Generation X (1996),Movie,Action|Adventure|Sci-Fi,,,Not provided by MovieLens,0.6808,Favorite-title/content fallback,Title + genres + tags; no plot supplied
9,6365,"Matrix Reloaded, The (2003)",Movie,Action|Adventure|Sci-Fi|Thriller|IMAX,,,Not provided by MovieLens,0.6786,Favorite-title/content fallback,Title + genres + tags; no plot supplied


,movieId,title,media_type,genres
1939,2571,"Matrix, The (1999)",Movie,Action|Sci-Fi|Thriller
4351,6365,"Matrix Reloaded, The (2003)",Movie,Action|Adventure|Sci-Fi|Thriller|IMAX
4639,6934,"Matrix Revolutions, The (2003)",Movie,Action|Adventure|Sci-Fi|Thriller|IMAX
5669,27660,"Animatrix, The (2003)",Movie,Action|Animation|Drama|Sci-Fi


## Because You Watched and natural-language search

In [7]:
display(engine.because_you_watched(1,n=10))
display(engine.search('A science fiction adventure about space travel',n=10))
# With a real extra_media.csv attached:
# display(engine.search('A mystery series',media_type='TV Show'))


,movieId,title,media_type,genres,description,tags,platform,score,reason,text_source,shared_genres
0,3114,Toy Story 2 (1999),Movie,Adventure|Animation|Children|Comedy|Fantasy,,Disney; Pixar; Tom Hanks; animation; funny; or...,Not provided by MovieLens,0.8186,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy, Fantasy"
1,78499,Toy Story 3 (2010),Movie,Adventure|Animation|Children|Comedy|Fantasy|IMAX,,,Not provided by MovieLens,0.7582,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy, Fantasy"
2,2355,"Bug's Life, A (1998)",Movie,Adventure|Animation|Children|Comedy,,Pixar,Not provided by MovieLens,0.7110,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy"
3,239,"Goofy Movie, A (1995)",Movie,Animation|Children|Comedy|Romance,,,Not provided by MovieLens,0.6771,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Animation, Children, Comedy"
4,145935,"Peanuts Movie, The (2015)",Movie,Adventure|Animation|Children|Comedy,,,Not provided by MovieLens,0.6756,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy"
5,108932,The Lego Movie (2014),Movie,Action|Adventure|Animation|Children|Comedy|Fan...,,cheeky; clever; colorful; feel-good; fun; imag...,Not provided by MovieLens,0.6795,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy, Fantasy"
6,2294,Antz (1998),Movie,Adventure|Animation|Children|Comedy|Fantasy,,,Not provided by MovieLens,0.6591,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy, Fantasy"
7,103755,Turbo (2013),Movie,Adventure|Animation|Children|Comedy|Fantasy,,,Not provided by MovieLens,0.6487,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy, Fantasy"
8,136016,The Good Dinosaur (2015),Movie,Adventure|Animation|Children|Comedy|Fantasy,,,Not provided by MovieLens,0.6440,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Animation, Children, Comedy, Fantasy"
9,3440,Teenage Mutant Ninja Turtles III (1993),Movie,Action|Adventure|Children|Comedy|Fantasy,,,Not provided by MovieLens,0.6326,MiniLM text similarity + genre overlap,Title + genres + tags; no plot supplied,"Adventure, Children, Comedy, Fantasy"


,movieId,title,media_type,genres,description,tags,platform,score,reason,text_source
0,4812,SpaceCamp (1986),Movie,Adventure|Sci-Fi,,NASA; space,Not provided by MovieLens,0.5660,MiniLM semantic search,Title + genres + tags; no plot supplied
1,121007,Space Buddies (2009),Movie,Adventure|Children|Fantasy|Sci-Fi,,,Not provided by MovieLens,0.5361,MiniLM semantic search,Title + genres + tags; no plot supplied
2,166526,The Space Between Us (2016),Movie,Adventure|Sci-Fi,,,Not provided by MovieLens,0.5189,MiniLM semantic search,Title + genres + tags; no plot supplied
3,68358,Star Trek (2009),Movie,Action|Adventure|Sci-Fi|IMAX,,Simon Pegg; future; lack of development; lack ...,Not provided by MovieLens,0.5162,MiniLM semantic search,Title + genres + tags; no plot supplied
4,60514,Journey to the Center of the Earth (2008),Movie,Action|Adventure|Sci-Fi,,,Not provided by MovieLens,0.5095,MiniLM semantic search,Title + genres + tags; no plot supplied
5,7743,Explorers (1985),Movie,Adventure|Children|Sci-Fi,,,Not provided by MovieLens,0.5081,MiniLM semantic search,Title + genres + tags; no plot supplied
6,1831,Lost in Space (1998),Movie,Action|Adventure|Sci-Fi,,,Not provided by MovieLens,0.5050,MiniLM semantic search,Title + genres + tags; no plot supplied
7,3927,Fantastic Voyage (1966),Movie,Adventure|Sci-Fi,,,Not provided by MovieLens,0.5011,MiniLM semantic search,Title + genres + tags; no plot supplied
8,71106,Frequently Asked Questions About Time Travel (...,Movie,Comedy|Sci-Fi,,,Not provided by MovieLens,0.4948,MiniLM semantic search,Title + genres + tags; no plot supplied
9,6239,Journey to the Center of the Earth (1959),Movie,Adventure|Children|Sci-Fi,,,Not provided by MovieLens,0.4945,MiniLM semantic search,Title + genres + tags; no plot supplied


## Top Picks for an existing profile and Most Popular
User IDs are anonymous MovieLens participants, not your account. Most Popular uses the latest observed dataset time by default, not today's date. Without view/release files, it is a **historical rating-activity proxy**. Its weights are 70% normalized log recent unique raters + 30% Bayesian-smoothed quality. With both real viewing and episode/movie release events, weights become 55% recency + 25% quality + 20% early demand. Demand counts unique viewers in the first 48 hours and decays by release age; it is never inferred from rating time.

In [8]:
USER_ID = int(data['selected_users'].userId.iloc[0])
display(engine.top_picks(USER_ID,n=10))
display(engine.most_popular(n=10))


,movieId,title,media_type,genres,description,tags,platform,score,reason,text_source
0,858,"Godfather, The (1972)",Movie,Crime|Drama,,Mafia,Not provided by MovieLens,0.8860,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
1,58559,"Dark Knight, The (2008)",Movie,Action|Crime|Drama|IMAX,,dark; gritty; psychology; superhero,Not provided by MovieLens,0.8800,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
2,318,"Shawshank Redemption, The (1994)",Movie,Crime|Drama,,Morgan Freeman; Stephen King; prison; wrongful...,Not provided by MovieLens,0.8751,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
3,177593,"Three Billboards Outside Ebbing, Missouri (2017)",Movie,Crime|Drama,,,Not provided by MovieLens,0.8566,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
4,1704,Good Will Hunting (1997),Movie,Drama|Romance,,heartwarming; inspirational,Not provided by MovieLens,0.8572,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
5,1266,Unforgiven (1992),Movie,Drama|Western,,revenge,Not provided by MovieLens,0.8579,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
6,142488,Spotlight (2015),Movie,Thriller,,,Not provided by MovieLens,0.8606,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
7,3275,"Boondock Saints, The (2000)",Movie,Action|Crime|Drama|Thriller,,,Not provided by MovieLens,0.8556,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
8,1193,One Flew Over the Cuckoo's Nest (1975),Movie,Drama,,emotional; jack nicholson; mental illness,Not provided by MovieLens,0.8491,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied
9,7153,"Lord of the Rings: The Return of the King, The...",Movie,Action|Adventure|Drama|Fantasy,,Adventure; Tolkein; Tolkien; ensemble cast; fa...,Not provided by MovieLens,0.8503,ALS + MiniLM + genres + popularity,Title + genres + tags; no plot supplied


,movieId,title,score,recent_unique_users,rating_count,bayesian_rating,keen_score,signal,as_of
0,79132,Inception (2010),0.939226,10,140,3.987107,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
1,2571,"Matrix, The (1999)",0.921205,9,272,4.150470,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
2,187593,Deadpool 2 (2018),0.918320,10,12,3.638661,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
3,7153,"Lord of the Rings: The Return of the King, The...",0.915722,9,182,4.059095,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
4,4993,"Lord of the Rings: The Fellowship of the Ring,...",0.884041,8,196,4.043691,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
5,5952,"Lord of the Rings: The Two Towers, The (2002)",0.879508,8,186,3.968142,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
6,8961,"Incredibles, The (2004)",0.868596,8,123,3.786274,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
7,122916,Thor: Ragnarok (2017),0.867075,8,20,3.760929,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
8,122912,Avengers: Infinity War - Part I (2018),0.863124,8,13,3.695065,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00
9,2959,Fight Club (1999),0.858815,7,214,4.196313,NaN,Historical unique raters (not viewers),2018-09-24 14:27:30+00:00


## Interactive recommendation controls
If widgets are unavailable in your notebook viewer, the function calls above provide identical functionality.

In [9]:
import ipywidgets as widgets
options = [(f'{row.title} [ID {row.movieId}]',int(row.movieId)) for row in engine.catalog.itertuples()]
favorite_controls = [widgets.Combobox(options=tuple(label for label,_ in options),
                                     value=next(label for label,mid in options if mid==choice),
                                     description=f'Favorite {i+1}',ensure_option=True,
                                     layout=widgets.Layout(width='700px'))
                     for i,choice in enumerate(FAVORITES)]
lookup = dict(options)
favorite_button=widgets.Button(description='Recommend favorites',button_style='primary')
query=widgets.Text(value='An exciting science fiction adventure',description='Request:',layout=widgets.Layout(width='700px'))
search_button=widgets.Button(description='Search titles')
user_control=widgets.Dropdown(options=[int(u) for u in data['selected_users'].userId],description='User:')
user_button=widgets.Button(description='Top Picks for You')
output=widgets.Output()
def render(action):
    with output:
        output.clear_output(wait=True)
        try: display(action())
        except Exception as error: print(str(error))
favorite_button.on_click(lambda _:render(lambda:engine.from_three_favorites([lookup[c.value] for c in favorite_controls])))
search_button.on_click(lambda _:render(lambda:engine.search(query.value)))
user_button.on_click(lambda _:render(lambda:engine.top_picks(user_control.value)))
display(widgets.VBox(favorite_controls+[favorite_button,query,search_button,user_control,user_button,output]))


## Collect new viewing feedback (optional)
Only call this after a user reports watching the selected title. It saves user-reported activity, never simulated telemetry. Completion >=80% contributes to the MiniLM profile, and watched titles are excluded immediately. New viewing records do not retrain the explicit-rating ALS model automatically; a future batch implicit-ALS update can learn from these logs once sufficient real activity has accumulated. Save the collected CSV before ending a Kaggle session and attach it through WATCH_EVENTS next time.

```python
# Example syntax only; this call is intentionally not executed:
engine.record_watch(user_id=10001, movie_id=1, completion=0.9)
display(engine.top_picks(user_id=10001))
```
No watch records are generated by Run All.

## Export results and database

In [10]:
panels = engine.export(USER_ID)
assert data['ratings'].userId.nunique() == 600
assert engine.E.shape == (len(engine.catalog),384)
assert not set(panels['three_favorites'].movieId) & set(FAVORITES)
assert not set(panels['top_picks'].movieId) & engine.seen.get(USER_ID,set())
assert 1 not in set(panels['because_you_watched'].movieId)
assert all(table.movieId.is_unique for table in panels.values())
assert engine.top_picks(user_id=-999,n=5).score.notna().all()
assert all(engine.test_metrics.evaluated_users > 0)
assert all(b <= a + 1e-6 for a,b in zip(engine.als_training_objective,engine.als_training_objective[1:]))
print('Passed: 600 users, MiniLM dimensions, seed/rated-title exclusions, unique recommendations, cold-user fallback.')
print('Saved CSV results, selected user IDs, embeddings, ALS factors and SQLite database to:',OUTPUT.resolve())


Passed: 600 users, MiniLM dimensions, seed/rated-title exclusions, unique recommendations, cold-user fallback.
Saved CSV results, selected user IDs, embeddings, ALS factors and SQLite database to: /kaggle/working/movielens_outputs


## Sources and limits
- GroupLens small dataset: https://grouplens.org/datasets/movielens/
- Dataset README: https://files.grouplens.org/datasets/movielens/ml-latest-small-README.html
- MiniLM model card: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
- Spark ALS: https://spark.apache.org/docs/latest/ml-collaborative-filtering.html
- VRV design inspiration: https://medium.com/crunchyroll/behind-vrvs-recommendation-engine-fec5cda717d0
- Citation: F. Maxwell Harper and Joseph A. Konstan (2015). The MovieLens Datasets: History and Context. ACM TiiS 5(4), Article 19. https://doi.org/10.1145/2827872

The code is for education and development; follow the supplied MovieLens license for data use. Do not imply GroupLens endorsement. Keep the original ZIP/README with your private project. This deliverable does not redistribute the raw dataset. Model scores are not confidence probabilities. MiniLM truncates long inputs at its token limit; concise descriptions work best. Weights are project design choices; choose new weights using validation only. No verified current streaming availability is provided. Extra descriptions improve evidence available to semantic matching, but better measured recommendations are not guaranteed. This notebook is not a deployed API.

In [11]:
from pathlib import Path

list(Path("/kaggle/input").rglob("ml-latest-small.zip"))

[]